# 04 — Claim Severity: Baseline and Gamma GLM

**Phase 6**, step 1 of 4: build the claim-level modelling table, fit a trivial baseline, and fit the first Gamma GLM.

Severity asks a different question from frequency: not "how often does a policy claim" but "given a claim happened, how much did it cost." That conditioning is not just conceptual -
it changes what the training data even looks like. A policy with zero claims has **no severity observation at all** (not a severity of zero), so this notebook works from the claim-level severity table (26,444 claims), not the 678,013-row policy table.

Phase 3's severity EDA already set expectations here: no rating factor showed a convincing univariate effect on severity, and the distribution is extremely skewed (top 1% of claims hold 38% of total value). This phase checks whether a proper multivariate model finds more than the univariate view did - but going in, a strong result is not expected.

In [1]:
import numpy as np
import pandas as pd

from auto_pricing.severity import (
    SEVERITY_FORMULA,
    baseline_severity,
    build_severity_table,
    fit_gamma_glm,
    predict_severity,
)

sev = pd.read_parquet("../data/processed/severity_clean.parquet")
model_table = pd.read_parquet("../data/processed/model_table.parquet")

severity_table = build_severity_table(sev, model_table)
print(f"severity table: {severity_table.shape}")
print(severity_table["split"].value_counts())

severity table: (26444, 20)
split
train         18523
test           3974
validation     3947
Name: count, dtype: int64


**18,523 / 3,947 / 3,974** claims across train/validation/test - matching *exactly* the claim counts `scripts/split_data.py` printed back in Phase 4 step 1. That match confirms the split-inheritance mechanism worked correctly: each claim was assigned to its **policy's** split (via the `IDpol` join), never split independently at the claim level - the same leakage-prevention rule from Phase 4, applied one level down.

In [2]:
train = severity_table[severity_table["split"] == "train"].copy()
print(f"train claims: {len(train):,}")

train claims: 18,523


## 1. Baseline severity

In [3]:
baseline = baseline_severity(train)
print(f"baseline (mean) severity: \u20ac{baseline:,.2f}")

baseline (mean) severity: €2,375.68


The simplest possible severity model: predict this one number for every claim, regardless of who or what caused it. Any real model needs to beat this - but Phase 3 already suggests it might not beat it by much.

## 2. Gamma GLM

In [4]:
print(SEVERITY_FORMULA)

ClaimAmount ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) + AreaOrdinal + LogDensity + VehGasBinary + C(RegionGrouped) + C(VehBrandGrouped) + VehPower


In [5]:
result = fit_gamma_glm(train)
print(f"converged: {result.converged}")
print(f"parameters: {len(result.params)}")
print(f"deviance: {result.deviance:.1f}")
print(f"dispersion (scale) estimate: {result.scale:.2f}")

converged: True
parameters: 48
deviance: 29265.2
dispersion (scale) estimate: 21.92


A **Gamma** distribution with a **log link** is the standard choice for a continuous, positive, right-skewed outcome like claim cost - the same logic as Poisson for counts, matched to a different shape of data. No exposure offset here: unlike claim frequency, a claim's cost has no mechanical relationship to how long the policy was observed before the claim happened.

The large dispersion (scale) estimate (~22) reflects exactly the severe skew found in Phase 3 (skewness ~110) - a Gamma distribution's dispersion parameter directly controls how heavy its tail is, and this data has a genuinely heavy one.

### Does the full model actually beat the baseline?

In [6]:
null_result = fit_gamma_glm(train, formula="ClaimAmount ~ 1")  # intercept-only = the baseline, refit as a GLM for a fair deviance comparison

deviance_reduction = (null_result.deviance - result.deviance) / null_result.deviance
print(f"null (baseline-only) deviance: {null_result.deviance:.1f}")
print(f"full model deviance:           {result.deviance:.1f}")
print(f"deviance reduction: {deviance_reduction:.1%}")

null (baseline-only) deviance: 34346.0
full model deviance:           29265.2
deviance reduction: 14.8%


**A 14.8% deviance reduction** - real, but modest, exactly matching Phase 3's expectation that severity would be much harder to explain from policy characteristics than frequency was. For comparison, Phase 5's frequency model beat its own baseline far more decisively. This is not a failure of the modelling approach - it is consistent with the plan's own expectation (and scikit-learn's official insurance tutorial, the project's methodological reference point) that severity is intrinsically harder to predict from these variables than frequency is.

### Sanity check: predicted vs. observed total (training data)

In [7]:
pred_train = predict_severity(result, train)
print(f"predicted total claim cost: \u20ac{pred_train.sum():,.2f}")
print(f"observed total claim cost:  \u20ac{train['ClaimAmount'].sum():,.2f}")
print(f"ratio: {pred_train.sum() / train['ClaimAmount'].sum():.4f}")

predicted total claim cost: €42,719,139.91
observed total claim cost:  €44,004,810.78
ratio: 0.9708


Off by about 2.9% - and this is expected, not a bug, for the same reason Phase 5's Negative Binomial model didn't exactly match its training total either. Only a Poisson GLM with a canonical link has the mathematical guarantee of an exact match; Gamma's estimating equations don't share that specific property. A small, consistent gap like this is normal; the frequency model's earlier 2x+ discrepancy (before that bug was fixed) was not.

## 3. Summary of step 1

- Claim-level table built (26,444 claims), correctly inheriting each claim's policy's split   assignment - verified against Phase 4's own split counts, not just assumed correct.
- Baseline severity: \u20ac2,375.68 (training mean).
- Gamma GLM fit successfully; the full feature set improves on the baseline by **14.8%   deviance reduction** - a real but modest improvement, consistent with Phase 3's finding   that severity has little relationship to the available rating factors.
- Predicted and observed training totals are close (ratio 0.971), consistent with a properly   fit model of this type.

### Next
Step 2: a lognormal alternative model, and the large-loss sensitivity analysis Phase 3's cleaning policy explicitly deferred to this phase (does the model's story change if the largest few claims are excluded?).

## 4. Lognormal alternative model

A common alternative to a Gamma GLM: take `log(ClaimAmount)`, fit ordinary regression on it, then exponentiate predictions back to the original scale. Done naively, this is a documented mistake (the plan explicitly warns against "taking logs and fitting OLS without addressing retransformation") - and it is not a small effect, as the next few cells show directly on this project's own data.

In [8]:
from auto_pricing.severity import fit_lognormal_model, predict_lognormal_severity

lognormal_result, smearing_factor = fit_lognormal_model(train)
print(f"smearing factor: {smearing_factor:.4f}")

smearing factor: 2.4771


### Why exponentiating a log-scale prediction directly is wrong

For a right-skewed variable, `exp(E[log Y])` gives the **median** of Y, not its **mean** - because the mean of a skewed distribution always sits above its median. Naively exponentiating a log-regression's prediction systematically *understates* expected cost. Duan's smearing estimator (the average of `exp(residual)` on the training data) corrects for this without assuming the residuals are exactly normally distributed.

In [9]:
naive_pred = np.exp(lognormal_result.predict(train))  # the mistake, on purpose, for contrast
corrected_pred = predict_lognormal_severity(lognormal_result, smearing_factor, train)

observed_total = train["ClaimAmount"].sum()
print(f"observed total:              \u20ac{observed_total:,.2f}")
print(f"naive predicted total:       \u20ac{naive_pred.sum():,.2f}  ({naive_pred.sum()/observed_total:.1%} of observed)")
print(f"corrected predicted total:   \u20ac{corrected_pred.sum():,.2f}  ({corrected_pred.sum()/observed_total:.1%} of observed)")

observed total:              €44,004,810.78
naive predicted total:       €17,625,306.59  (40.1% of observed)
corrected predicted total:   €43,658,860.17  (99.2% of observed)


**The naive version predicts only ~40% of the true total claim cost - understating it by 60%.** The smearing-corrected version predicts ~99%, essentially exact. This isn't a subtle statistical nuance to footnote - it's the difference between a severity model that is roughly right and one that is off by more than half, from the exact same underlying fit. `predict_lognormal_severity()` (`src/auto_pricing/severity.py`) applies the correction automatically and bundles the smearing factor with the fitted model specifically so this mistake isn't the easy path.

### Gamma GLM vs. lognormal: which fits better?

In [10]:
gamma_pred = predict_severity(result, train)
from sklearn.metrics import mean_gamma_deviance

gamma_dev = mean_gamma_deviance(train["ClaimAmount"], gamma_pred)
lognormal_dev = mean_gamma_deviance(train["ClaimAmount"], corrected_pred)
print(f"Gamma GLM - mean Gamma deviance on training data:      {gamma_dev:.4f}")
print(f"Lognormal (corrected) - mean Gamma deviance on training data: {lognormal_dev:.4f}")

Gamma GLM - mean Gamma deviance on training data:      1.5799
Lognormal (corrected) - mean Gamma deviance on training data: 1.8141


Both models are compared here using Gamma deviance (a fair, distribution-appropriate metric for positive right-skewed data) rather than plain RMSE, which the plan explicitly warns against relying on for this kind of outcome - a metric that treats a €10 error on a €100 claim the same as a €10 error on a €10,000 claim would be dominated entirely by the largest claims and say little about typical performance. A full validation-set comparison, and the champion decision between these two models, is step 3.

## 5. Large-loss sensitivity analysis

The analysis `reports/cleaning_policy.md` (Phase 3, anomaly 4) deferred to this phase: does the model's story change if the largest claims are excluded? Threshold from the **training split only** (top 1% by value), matching the fit-on-train discipline used throughout this project.

In [11]:
from auto_pricing.severity import large_loss_threshold, exclude_large_losses

threshold = large_loss_threshold(train, top_pct=0.01)
n_excluded = (train["ClaimAmount"] >= threshold).sum()
excluded_value = train.loc[train["ClaimAmount"] >= threshold, "ClaimAmount"].sum()

print(f"threshold (99th percentile): \u20ac{threshold:,.2f}")
print(f"claims excluded: {n_excluded} / {len(train)} ({n_excluded/len(train):.2%})")
print(f"value excluded: \u20ac{excluded_value:,.2f} ({excluded_value/train['ClaimAmount'].sum():.1%} of total)")

threshold (99th percentile): €16,795.72
claims excluded: 186 / 18523 (1.00%)
value excluded: €17,988,204.64 (40.9% of total)


**The top 1% of claims by count hold 41% of total claim value** - consistent with Phase 3's portfolio-wide finding (38%). A severity model's accuracy on total cost hinges on these 186 claims far more than on the other 18,337.

In [12]:
train_trimmed = exclude_large_losses(train, threshold)
trimmed_result = fit_gamma_glm(train_trimmed)

baseline_full = np.exp(result.params["Intercept"])
baseline_trimmed = np.exp(trimmed_result.params["Intercept"])
print(f"reference-category baseline severity, full model:    \u20ac{baseline_full:,.2f}")
print(f"reference-category baseline severity, top 1% excluded: \u20ac{baseline_trimmed:,.2f}")
print(f"ratio: {baseline_trimmed/baseline_full:.3f}")

reference-category baseline severity, full model:    €2,866.54
reference-category baseline severity, top 1% excluded: €1,336.44
ratio: 0.466


The baseline severity level drops by **53%** once the top 1% of claims are excluded - a direct, concrete illustration of why capping large claims by default (which Phase 3 declined to do) would have quietly halved the model's sense of typical severity, understating real tail risk rather than producing a cleaner model.

In [13]:
drivage_full = np.exp(result.params.filter(like="DrivAgeBand"))
drivage_trimmed = np.exp(trimmed_result.params.filter(like="DrivAgeBand"))
pd.DataFrame({"full_model": drivage_full, "top_1pct_excluded": drivage_trimmed}).round(3)

,full_model,top_1pct_excluded
C(DrivAgeBand)[T.23-29],0.280,0.872
C(DrivAgeBand)[T.30-39],0.272,0.907
C(DrivAgeBand)[T.40-49],0.261,0.928
C(DrivAgeBand)[T.50-59],0.260,0.944
C(DrivAgeBand)[T.60-69],0.281,0.952
C(DrivAgeBand)[T.70+],0.325,0.992


**This is the key finding of the sensitivity analysis.** In the full model, every DrivAge band 23+ appears to cost 65-75% *less* than the 18-22 reference - a dramatic pattern. Excluding just the top 1% of claims, that pattern **nearly disappears** (every band lands within roughly 5-13% of the reference). The apparent "young drivers have far more severe claims" relationship is not a broad pattern - it is the arithmetic consequence of a handful of very large claims happening to be concentrated in the 18-22 group specifically. This is a direct, proven instance of Phase 3's original warning: apparent severity-by-segment patterns trace back to a few large claims distorting a small sample, not a genuine relationship. `VehBrandGrouped[B11]` shows the identical signature (not shown here - full table in `reports/severity_large_loss_sensitivity.md`).

## 6. Summary of step 2, and the decision this leads to

- **Lognormal + Duan's smearing correction**: without it, the naive lognormal model predicts   only ~40% of true total cost - a large, real bias, not a footnote. With it, ~99%.
- **Large-loss sensitivity**: the top 1% of training claims hold 41% of total value.   Excluding them drops the baseline severity estimate by 53%, and makes the apparent   DrivAge severity effect (and VehBrandGrouped[B11]'s) almost entirely disappear.
- **Decision, consistent with `cleaning_policy.md` rule 4**: the primary model keeps using   the full, uncapped data - large losses are real and expected, not noise to remove. But   `DrivAgeBand`'s severity relativity is now known to be fragile, driven by a handful of   large claims rather than a broad pattern, and step 4's interpretation will flag this   explicitly rather than report the full model's dramatic age effect unqualified.

Full write-up: `reports/severity_large_loss_sensitivity.md`.

### Next
Step 3: evaluate Gamma vs. lognormal on the validation split (never touched for fitting) and select a champion.

## 7. Evaluation on the validation split, and champion selection

Everything so far was fit or diagnosed on the training split. This is the first time `split == "validation"` is touched, and only for evaluation. Full rationale for every decision below is in `reports/severity_model_selection.md`.

In [14]:
from auto_pricing.evaluation import gamma_deviance, observed_to_expected_ratio, severity_calibration_by_decile

val = severity_table[severity_table["split"] == "validation"].copy()
print(f"validation claims: {len(val):,}")

validation claims: 3,947


In [15]:
baseline_pred_val = pd.Series([baseline] * len(val), index=val.index)
gamma_pred_val = predict_severity(result, val)
lognormal_pred_val = predict_lognormal_severity(lognormal_result, smearing_factor, val)

for name, pred in [("baseline", baseline_pred_val), ("gamma", gamma_pred_val), ("lognormal", lognormal_pred_val)]:
    dev = gamma_deviance(val["ClaimAmount"], pred)
    oe = observed_to_expected_ratio(val["ClaimAmount"], pred)
    print(f"{name:<12} deviance={dev:.4f}  observed/expected={oe:.4f}")

baseline     deviance=1.5690  observed/expected=0.8436
gamma        deviance=1.5724  observed/expected=0.8755
lognormal    deviance=1.5273  observed/expected=0.8496


**A real, checked surprise: the full Gamma model (1.5724) is *worse* than simply predicting the training mean for every claim (1.5690)** - despite improving training deviance by 14.8% (step 1). This is genuine overfitting, not a rounding artifact. Lognormal (1.5273) is the best of the three.

### Where is Gamma's overfitting coming from?

In [16]:
reduced_formula = (
    "ClaimAmount ~ C(DrivAgeBand) + C(VehAgeBand) + C(BonusMalusBand) "
    "+ AreaOrdinal + LogDensity + VehGasBinary + VehPower"
)

gamma_reduced = fit_gamma_glm(train, formula=reduced_formula)
gamma_reduced_pred = predict_severity(gamma_reduced, val)
print(f"Gamma, no Region/VehBrand ({len(gamma_reduced.params)} params): "
      f"deviance={gamma_deviance(val['ClaimAmount'], gamma_reduced_pred):.4f}")

lognormal_reduced, smearing_reduced = fit_lognormal_model(train, formula=reduced_formula)
lognormal_reduced_pred = predict_lognormal_severity(lognormal_reduced, smearing_reduced, val)
print(f"Lognormal, no Region/VehBrand ({len(lognormal_reduced.params)} params): "
      f"deviance={gamma_deviance(val['ClaimAmount'], lognormal_reduced_pred):.4f}")

Gamma, no Region/VehBrand (22 params): deviance=1.5567
Lognormal, no Region/VehBrand (22 params): deviance=1.5515


Dropping `Region`/`VehBrand` (26 of 48 parameters, already known from Phase 3 to have no reliable univariate severity relationship) **improves** Gamma's validation deviance (1.5724 -> 1.5567, now beating the baseline) but **worsens** lognormal's (1.5273 -> 1.5515) - the opposite reaction to the same change. Gamma's fitting weights errors by the *square* of the predicted value, so a handful of large, noisy claims in specific regions/brands pull those coefficients toward overfit values; fitting on the log scale (lognormal) compresses that same leverage, so those parameters add a little real signal instead of mostly noise. Lognormal's full-formula version remains the best result overall.

### Calibration by decile: where does Gamma actually break down?

In [17]:
gamma_calib = severity_calibration_by_decile(val["ClaimAmount"], gamma_pred_val)
lognormal_calib = severity_calibration_by_decile(val["ClaimAmount"], lognormal_pred_val)

print("Gamma, top decile (highest predicted severity):")
print(gamma_calib.iloc[-1])
print()
print("Lognormal, top decile:")
print(lognormal_calib.iloc[-1])

Gamma, top decile (highest predicted severity):
decile               9.000000
n_claims           395.000000
observed_mean     2962.837747
predicted_mean    5842.046330
Name: 9, dtype: float64

Lognormal, top decile:
decile               9.000000
n_claims           395.000000
observed_mean     2523.342608
predicted_mean    3121.274696
Name: 9, dtype: float64


Gamma's highest-predicted-severity decile over-predicts by **~97%** (observed €2,963 vs. predicted €5,842) - a portfolio-level observed/expected ratio close to 1 can hide exactly this kind of concentrated miscalibration in the one segment a pricing model most needs to get right. Lognormal's worst decile miss is proportionally much smaller.

### A secondary check: MAE tells a different, and expected, story

In [18]:
from sklearn.metrics import mean_absolute_error

for name, pred in [("baseline", baseline_pred_val), ("gamma", gamma_pred_val), ("lognormal", lognormal_pred_val)]:
    mae = mean_absolute_error(val["ClaimAmount"], pred)
    print(f"{name:<12} MAE=\u20ac{mae:,.2f}")

baseline     MAE=€2,116.26
gamma        MAE=€2,089.09
lognormal    MAE=€2,108.52


By plain MAE, Gamma edges out lognormal slightly - all three are within 1.3% of each other. This is not a contradiction; it is the plan's own warning ("judging performance only through RMSE") playing out directly: MAE weights every euro equally regardless of scale, so it barely registers the large *relative* miscalibration Gamma shows in its top decile. The much larger, more diagnostic gap in deviance and decile calibration is what should - and does - decide the champion here.

### Does the champion resolve step 2's fragility concern?

In [19]:
lognormal_full_trimmed, _ = fit_lognormal_model(train_trimmed)
lognormal_drivage_full = np.exp(lognormal_result.params.filter(like="DrivAgeBand"))
lognormal_drivage_trimmed = np.exp(lognormal_full_trimmed.params.filter(like="DrivAgeBand"))
pd.DataFrame({"full": lognormal_drivage_full, "top_1pct_excluded": lognormal_drivage_trimmed}).round(3)

,full,top_1pct_excluded
C(DrivAgeBand)[T.23-29],0.872,0.918
C(DrivAgeBand)[T.30-39],0.938,0.989
C(DrivAgeBand)[T.40-49],1.043,1.107
C(DrivAgeBand)[T.50-59],1.084,1.152
C(DrivAgeBand)[T.60-69],1.098,1.172
C(DrivAgeBand)[T.70+],1.259,1.312


Yes - and this is not a coincidence, it's the same mechanism as the overfitting finding above. The lognormal champion's `DrivAgeBand` relativities are modest in the full model (0.87-1.26) and barely move when the top 1% of claims are excluded (0.92-1.31) - nothing like Gamma's near-total reversal (0.28 -> 0.87 for the same band, step 2). Fitting on the log scale is inherently more resistant to a few large claims dominating a coefficient, which is exactly why the same model both generalizes better *and* is more robust to the large-loss sensitivity check.

## 8. Champion: Lognormal (Duan smearing-corrected), full formula

Best validation deviance among every candidate tested (baseline, Gamma full/reduced, lognormal full/reduced), the most resilient decile calibration, and the most stable `DrivAgeBand` relativities under the large-loss sensitivity check. Not the "textbook default" choice for this kind of skewed positive outcome (Gamma usually is) - the choice the validation evidence actually supports.

**Caveat carried forward, not discarded**: even the champion only modestly beats the trivial baseline (1.5273 vs. 1.5690 deviance, ~2.7% better). This is consistent with, not a contradiction of, Phase 3's original finding that severity has little genuine relationship to the available rating factors - this model's real value is a modest, honest improvement over "just use the average," not strong predictive power.

Full decision record: `reports/severity_model_selection.md`.

### Next
Step 4: interpretation - relativity tables for the champion model, a plain-language business narrative, and the Phase 6 wrap-up.

## 9. Interpretation: relativities for the champion (lognormal) model

Full tables and narrative: `reports/severity_relativities.md`. Before trusting confidence intervals, the champion's residual assumptions were checked - the same discipline Phase 5 applied to Poisson's understated standard errors.

In [20]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import jarque_bera

jb_stat, jb_p, skew, kurtosis = jarque_bera(lognormal_result.resid)
bp_stat, bp_p, _, _ = het_breuschpagan(lognormal_result.resid, lognormal_result.model.exog)
print(f"Jarque-Bera (residual normality): stat={jb_stat:.1f}  p={jb_p:.2e}")
print(f"Breusch-Pagan (constant variance): stat={bp_stat:.1f}  p={bp_p:.2e}")

Jarque-Bera (residual normality): stat=10034.2  p=0.00e+00
Breusch-Pagan (constant variance): stat=234.0  p=1.19e-26


Both assumptions plain OLS inference relies on are rejected - residuals are not normal (heavy-tailed, kurtosis 6.5) and not constant-variance. **Robust (HC3) standard errors are used for every confidence interval below**, not the model's own naive ones.

In [21]:
robust = lognormal_result.get_robustcov_results(cov_type="HC3")

sev_relativities = pd.DataFrame({
    "coefficient": lognormal_result.params,
    "robust_se": pd.Series(robust.bse, index=lognormal_result.params.index),
})
sev_relativities["relativity"] = np.exp(sev_relativities["coefficient"])
sev_relativities["ci_low"] = np.exp(sev_relativities["coefficient"] - 1.96 * sev_relativities["robust_se"])
sev_relativities["ci_high"] = np.exp(sev_relativities["coefficient"] + 1.96 * sev_relativities["robust_se"])
sev_relativities["significant"] = ~((sev_relativities["ci_low"] < 1) & (sev_relativities["ci_high"] > 1))

sev_relativities.filter(like="BonusMalusBand", axis=0).round(3)

,coefficient,robust_se,relativity,ci_low,ci_high,significant
C(BonusMalusBand)[T.51-59],0.061,0.029,1.062,1.004,1.124,True
C(BonusMalusBand)[T.60-79],0.201,0.022,1.223,1.171,1.278,True
C(BonusMalusBand)[T.80-99],0.195,0.030,1.215,1.146,1.288,True
C(BonusMalusBand)[T.100-129],0.372,0.034,1.451,1.358,1.550,True
C(BonusMalusBand)[T.130+],0.508,0.115,1.663,1.328,2.081,True


Every BonusMalus band significant, monotonically increasing to **1.66x** at the worst score. Phase 3's univariate EDA found this looked like noise (flat median, a mean distorted by large claims) - a proper multivariate model on a large-claim-resistant scale finds a real relationship after all. Checked directly against the sensitivity test used in step 2/3 (rather than assumed robust just because lognormal generally handled DrivAge well):

In [22]:
bonusmalus_full = np.exp(lognormal_result.params.filter(like="BonusMalusBand"))
bonusmalus_trimmed = np.exp(lognormal_full_trimmed.params.filter(like="BonusMalusBand"))
pd.DataFrame({"full": bonusmalus_full, "top_1pct_excluded": bonusmalus_trimmed}).round(3)

,full,top_1pct_excluded
C(BonusMalusBand)[T.51-59],1.062,1.050
C(BonusMalusBand)[T.60-79],1.223,1.219
C(BonusMalusBand)[T.80-99],1.215,1.214
C(BonusMalusBand)[T.100-129],1.451,1.444
C(BonusMalusBand)[T.130+],1.663,1.555


Barely moves (1.663 -> 1.555 at the extreme, everything else nearly identical) - this finding is genuine, not another large-claim artifact like DrivAge partially was.

In [23]:
sev_relativities.filter(like="VehAgeBand", axis=0).round(3)

,coefficient,robust_se,relativity,ci_low,ci_high,significant
C(VehAgeBand)[T.1-2],-0.001,0.042,0.999,0.920,1.084,False
C(VehAgeBand)[T.3-5],-0.034,0.043,0.966,0.888,1.051,False
C(VehAgeBand)[T.6-9],-0.014,0.043,0.986,0.906,1.074,False
C(VehAgeBand)[T.10-14],-0.086,0.044,0.918,0.843,1.000,False
C(VehAgeBand)[T.15-19],-0.017,0.050,0.983,0.891,1.085,False
C(VehAgeBand)[T.20+],-0.085,0.084,0.918,0.778,1.083,False


**No VehAge band is significant** - a striking contrast with Phase 5, where VehAge was one of the *strongest* frequency predictors (the new-vehicle spike). How often a vehicle claims and how expensive that claim is, once it happens, appear to be governed by different things.

In [24]:
sev_relativities.loc[["AreaOrdinal", "LogDensity", "VehGasBinary", "VehPower"]].round(4)

,coefficient,robust_se,relativity,ci_low,ci_high,significant
AreaOrdinal,-0.0330,0.0253,0.9675,0.9207,1.0167,False
LogDensity,0.0190,0.0188,1.0192,0.9822,1.0575,False
VehGasBinary,-0.0180,0.0172,0.9822,0.9496,1.0159,False
VehPower,0.0033,0.0045,1.0033,0.9945,1.0121,False


None significant for severity - another contrast with frequency, where `VehPower` and `VehGasBinary` were both significant contributors.

In [25]:
baseline_severity_eur = np.exp(lognormal_result.params["Intercept"]) * smearing_factor
print(f"reference-category baseline severity: \u20ac{baseline_severity_eur:,.2f}")

reference-category baseline severity: €1,881.91


## 10. Phase 6 complete - business narrative

Severity tells a genuinely different, and much sparser, story than frequency. **BonusMalus is the one clear, robust driver of claim cost**, not just claim count - a driver's own claims history predicts both how often they claim and how expensive those claims are. Age plays a smaller, partly uncertain role (youngest-adult claims somewhat cheaper, oldest-band claims somewhat more expensive, but less robust than BonusMalus's result). Everything else that mattered for frequency - vehicle age, vehicle power, fuel type, population density, area type - shows **no meaningful relationship to severity** here. A policy's rating factors mostly explain *whether* it claims, not *how expensive* that claim will be when it does.

Full tables, all caveats, and the champion decision: `reports/severity_model_selection.md` and `reports/severity_relativities.md`. Phase 6 is complete; Phase 7 (combining frequency and severity into pure premium, and reconciling them against the actual observed loss cost) is next.